# Crypto.Net · 01 — Dompet dan kunci

*Dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*

> **Aman dijalankan.** Notebook ini hanya membaca jaringan publik dan menandatangani secara offline; tidak pernah mem-broadcast transaksi. Kunci yang dibuat di sini adalah kunci demo — jangan pernah mengirim dana sungguhan ke sana.

Buka di VS Code dengan ekstensi **Polyglot Notebooks** (atau Jupyter dengan .NET Interactive) lalu jalankan sel secara berurutan. [English version](./01-wallet-and-keys.en.ipynb)

Yang akan Anda lakukan: membuat frasa pemulihan BIP-39, menurunkan akun di lima chain darinya, mengekspor extended public key, mengenkripsi kunci ke keystore Web3, dan membandingkan inti Rust dengan mesin C# managed.

In [ ]:
#r "nuget: Crypto.Net.Extensions, 1.0.0"

In [ ]:
using Crypto.Net.Core;
using Crypto.Net.Native;
using Crypto.Net.Wallet;
using Crypto.Net.Evm;
using Crypto.Net.Bitcoin;
using Crypto.Net.Solana;
using Crypto.Net.Polkadot;
using Crypto.Net.Cosmos;
using Crypto.Net.Extensions;

$"Rust core loaded: {NativeLoader.IsNativeAvailable} · ABI 0x{NativeLoader.AbiVersion:X8} · {NativeLoader.RuntimeIdentifier}"

## Frasa pemulihan (BIP-39)

12–24 kata mengodekan 128–256 bit entropi ditambah checksum. `Mnemonic.Validate` memeriksa daftar kata sekaligus checksum-nya.

In [ ]:
var phrase = Mnemonic.Generate(12);
Console.WriteLine(phrase);
Console.WriteLine($"valid: {Mnemonic.Validate(phrase)}");
Console.WriteLine($"one wrong word: {Mnemonic.Validate(phrase.Replace(phrase.Split(' ')[0], "zoo"))}");

Bagian selanjutnya memakai frasa uji BIP-39 yang publik agar hasilnya bisa direproduksi — dan bisa Anda bandingkan dengan MetaMask, Phantom, Polkadot.js atau Keplr.

In [ ]:
const string TestPhrase = "abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon about";
byte[] seed = Mnemonic.ToSeed(TestPhrase);
Convert.ToHexStringLower(seed)

## Satu frasa, semua chain

`HdWallet` menyimpan tiga root: BIP-32 (secp256k1) untuk EVM, Bitcoin dan Cosmos, SLIP-10 (ed25519) untuk Solana, dan substrate-bip39 untuk Polkadot.

In [ ]:
var wallet = HdWallet.FromMnemonic(TestPhrase);

var eth  = wallet.GetEvmAccount(0, EvmChain.Ethereum);
var btc  = wallet.GetBitcoinAccount(BitcoinAddressType.SegWitP2WPKH);
var tap  = wallet.GetBitcoinAccount(BitcoinAddressType.TaprootP2TR);
var sol  = wallet.GetSolanaAccount(0);
var dot  = wallet.GetPolkadotAccount("", PolkadotChain.Polkadot);
var atom = wallet.GetCosmosAccount(0);

new[]
{
    new { Chain = "Ethereum",          Path = eth.DerivationPath,  Address = eth.Address.Value },
    new { Chain = "Bitcoin (SegWit)",  Path = btc.DerivationPath,  Address = btc.Address.Value },
    new { Chain = "Bitcoin (Taproot)", Path = tap.DerivationPath,  Address = tap.Address.Value },
    new { Chain = "Solana",            Path = sol.DerivationPath,  Address = sol.Address.Value },
    new { Chain = "Polkadot",          Path = "(root, sr25519)",   Address = dot.Address.Value },
    new { Chain = "Cosmos Hub",        Path = atom.DerivationPath, Address = atom.Address.Value },
}

## Extended key dan path kustom

`zpub`/`xpub` tingkat akun memungkinkan dompet watch-only menurunkan alamat penerimaan tanpa private key.

In [ ]:
Console.WriteLine($"BIP-84 zpub: {wallet.GetBitcoinAccountXpub(BitcoinAddressType.SegWitP2WPKH)}");

var key = wallet.DerivePath("m/44'/60'/0'/0/7");
Console.WriteLine($"m/44'/60'/0'/0/7 → {EvmAddress.FromPrivateKey(key.PrivateKey.Span)}");
Console.WriteLine($"depth {key.Depth}, parent fingerprint {key.ParentFingerprint:x8}");
key.Dispose();

## Keystore terenkripsi (Web3 Secret Storage v3)

Format yang sama dengan yang diimpor geth dan MetaMask. `KeyStoreKdf.Light` membuat demo ini cepat; gunakan `KeyStoreKdf.Standard` di aplikasi sungguhan.

In [ ]:
string json = eth.ExportKeyStore("correct horse battery staple", KeyStoreKdf.Light);
Console.WriteLine(json);

var restored = EvmAccount.FromKeyStore(json, "correct horse battery staple");
Console.WriteLine($"restored {restored.Address} — same account: {restored.Address == eth.Address}");
try { KeyStore.Decrypt(json, "wrong password"); }
catch (System.Security.Cryptography.CryptographicException e) { Console.WriteLine($"wrong password → {e.Message}"); }

## Dua mesin, byte identik

Setiap primitif berjalan di Rust atau C# managed. `UseBackend` mengganti mesin hanya untuk alur saat ini; keluarannya identik per byte.

In [ ]:
byte[] data = "Crypto.Net"u8.ToArray();
string rust, managed;
using (CryptoNative.UseBackend(CryptoBackend.Native))  rust    = Convert.ToHexStringLower(CryptoNative.Keccak256(data));
using (CryptoNative.UseBackend(CryptoBackend.Managed)) managed = Convert.ToHexStringLower(CryptoNative.Keccak256(data));
Console.WriteLine($"{rust}\n{managed}\nidentical: {rust == managed}");

CryptoBenchmark.Run(scale: 0.1)
    .Select(r => new { r.Name, Rust = Math.Round(r.NativeOpsPerSec), Managed = Math.Round(r.ManagedOpsPerSec), SpeedUp = Math.Round(r.Speedup, 1) })

Dispose rahasia setelah selesai.

In [ ]:
foreach (IDisposable d in new IDisposable[] { eth, btc, tap, sol, dot, atom, restored, wallet }) d.Dispose();
"disposed"